**Enrolment number(s):** XXXXXXXX

# Problem 2 — German census 2022

## 2.1 Import and data conventions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = "./data_census/"
# IDs are read as strings: they are identifiers, and numeric parsing would invite accidental arithmetic
census = pd.read_csv(DATA + "data_census.csv", sep=";", dtype={"MunicipalityID": str})
locations = pd.read_csv(DATA + "municipality-locations.csv", sep=";", dtype={"MunicipalityID": str})

AGES = ["<=3", "3-5", "6-14", "15-17", "18-24", "25-29", "30-39", "40-49", "50-64", "65-74", ">=75"]
census["Age"] = pd.Categorical(census["Age"], categories=AGES + ["Overall"], ordered=True)

census.head()

,Municipality,MunicipalityID,Age,Marital status,Gender,Count
0,Enzen,72325008032,40-49,Overall,Overall,3
1,Bernitt,130725252013,Overall,Married,Male,319
2,Leidenborn,72325001259,30-39,Divorced,Female,0
3,Schwielowsee,120690590590,50-64,Overall,Male,1450
4,Heßheim,73385006012,40-49,Overall,Female,175


**Checks.** The national total is stored as municipality `Overall` with ID `0`. It is the only
municipality with duplicated attribute combinations, where the duplicate carries a spurious `0`;
we keep the larger value. All other municipalities are unique per (Age, Marital status, Gender).

In [2]:
KEY = ["MunicipalityID", "Age", "Marital status", "Gender"]
print("duplicated keys per municipality:",
      list(census[census.duplicated(KEY, keep=False)]["MunicipalityID"].unique()))

census = (census.sort_values("Count", ascending=False)
          .drop_duplicates(KEY)
          .sort_values(KEY)
          .reset_index(drop=True))

germany = census[census["MunicipalityID"] == "0"]
muni = census[census["MunicipalityID"] != "0"]

print(f"{muni['MunicipalityID'].nunique()} municipalities, "
      f"{muni['Municipality'].nunique()} distinct names (names are not unique)")
print("municipalities without location:",
      len(set(muni["MunicipalityID"]) - set(locations["MunicipalityID"])))

duplicated keys per municipality: ['0']
10786 municipalities, 10759 distinct names (names are not unique)
municipalities without location: 0


**Totals vs. sums.** Masking means sub-group sums do not reproduce the `Overall` rows. We therefore
always read totals from the `Overall` rows rather than summing.

In [3]:
def select(df, age="Overall", marital="Overall", gender="Overall"):
    """Rows for one slice; pass None to keep an attribute resolved."""
    m = pd.Series(True, index=df.index)
    for col, val in [("Age", age), ("Marital status", marital), ("Gender", gender)]:
        if val is not None:
            m &= df[col] == val
    return df[m]

totals = select(muni).set_index("MunicipalityID")["Count"].rename("residents")
by_gender = select(muni, gender=None).query("Gender != 'Overall'") \
    .pivot(index="MunicipalityID", columns="Gender", values="Count")

gap = totals - by_gender.sum(axis=1)
print(f"national total (row 0): {select(germany)['Count'].item():,}")
print(f"sum of municipal totals: {totals.sum():,}")
print(f"municipalities where Male+Female != Overall: {(gap != 0).mean():.0%}, "
      f"max |gap| = {gap.abs().max()}")

national total (row 0): 82,719,540
sum of municipal totals: 82,711,282
municipalities where Male+Female != Overall: 69%, max |gap| = 11


In [4]:
mdf = (totals.to_frame()
       .join(by_gender)
       .join(locations.set_index("MunicipalityID")[["Municipality", "lon", "lat"]]))
mdf.describe()

,residents,Female,Male,lon,lat
count,1.078600e+04,1.078600e+04,1.078600e+04,10786.000000,10786.000000
mean,7.668393e+03,3.897855e+03,3.770544e+03,9.897349,50.861406
std,4.951918e+04,2.531932e+04,2.420118e+04,2.076123,1.959182
min,9.000000e+00,0.000000e+00,4.000000e+00,5.902304,47.408447
25%,6.552500e+02,3.270000e+02,3.272500e+02,8.045413,49.442822
50%,1.797000e+03,8.960000e+02,9.000000e+02,9.783509,50.463915
75%,5.505000e+03,2.782000e+03,2.736000e+03,11.461336,52.490700
max,3.596999e+06,1.839986e+06,1.757012e+06,14.988035,55.020026
